---

## NOTEBOOK 6 — Comparaison globale des modèles

---

### Plan du notebook

| Cellule | Section | Contenu |
|---------|---------|--------|
| C2 | 1. Configuration | Imports, seed, chemins |
| C3 | 2. Chargement CSV | Concat des 4 fichiers résultats protocole |
| C4 | 3. Tableau comparatif | Tous modèles côte à côte |
| C5 | 4. Barplot F1 macro | Classement visuel |
| C6 | 5. Analyse par genre | Recall par genre × modèle |
| C7 | 6. Matrices de confusion | Côte à côte (meilleurs modèles) |
| C8 | 7. Mismatch × modèle | Performance conditionnelle cohérents vs ambigus |
| C9 | 8. Radar chart | Profil multi-métrique par modèle |
| — | Analyse | Synthèse des résultats |
| — | Conclusion | Meilleur modèle, limites, recommandations |

---

### Objectif

Comparer les 5 modèles entraînés dans les notebooks précédents sur la tâche
de classification `genre_top` (8 genres, FMA Small, 351 features audio).
Même split GroupShuffleSplit artiste (seed=42, test_size=0.2) pour tous.

---

### Modèles comparés

| Notebook | Modèle | Auteur |
|----------|--------|--------|
| NB3 | Logistic Regression | Camille |
| NB3 | Random Forest | Camille |
| NB3BIS | XGBoost (GridSearch) | Diego |
| NB4 | CNN log-mel | Jayson |
| NB5 | XGBoost top-1 (mismatch) | Xia |

---

In [ ]:
# C2
# 1. Configuration

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')

SEED      = 42
TEST_SIZE = 0.2
np.random.seed(SEED)

BASE        = Path.cwd()
RESULTS_DIR = BASE / 'outputs' / 'resultats'
OUTPUT_DIR  = BASE / 'outputs' / 'comparaison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Python : {sys.version.split()[0]}')
print(f'pandas : {pd.__version__}')
print(f'RESULTS_DIR : {RESULTS_DIR}')

In [ ]:
# C3
# 2. Chargement des CSV résultats — tronc commun protocole (15 colonnes)

csv_files = {
    'NB3':    RESULTS_DIR / 'results_nb3.csv',
    'NB3BIS': RESULTS_DIR / 'results_nb3bis.csv',
    'NB4':    RESULTS_DIR / 'results_nb4.csv',
    'NB5':    RESULTS_DIR / 'results_nb5.csv',
}

dfs = []
for nb, path in csv_files.items():
    if path.exists():
        tmp = pd.read_csv(path)
        tmp['notebook'] = nb
        dfs.append(tmp)
        print(f'{nb:6s} : {path.name} — {len(tmp)} ligne(s) ✅')
    else:
        print(f'{nb:6s} : {path.name} — MANQUANT ⚠️')

df_all = pd.concat(dfs, ignore_index=True)
print(f'\nTotal : {len(df_all)} modèles chargés')
print(f'Colonnes : {list(df_all.columns)}')

In [ ]:
# C4
# 3. Tableau comparatif — tous modèles

# Colonnes d'affichage (tronc commun)
DISPLAY_COLS = ['notebook', 'model', 'f1_test', 'acc_test', 'bal_acc_test',
                'f1_cv_mean', 'f1_cv_std', 'duration_s', 'scaler']
display_cols = [c for c in DISPLAY_COLS if c in df_all.columns]

# Filtrer les lignes genre_top (exclure multi-label / top-k de NB5)
if 'task' in df_all.columns:
    mask_genre_top = df_all['task'].isna() | df_all['task'].str.contains('genre_top', na=True)
    df_compare = df_all[mask_genre_top].copy()
else:
    df_compare = df_all.copy()

df_compare = df_compare.sort_values('f1_test', ascending=False).reset_index(drop=True)

print('=== TABLEAU COMPARATIF — CLASSIFICATION GENRE_TOP ===')
print(df_compare[display_cols].to_string(index=False))

# Meilleur modèle
best = df_compare.iloc[0]
print(f'\nMeilleur modèle : {best["model"]} ({best["notebook"]}) — F1 macro = {best["f1_test"]}')

In [ ]:
# C5
# 4. Barplot F1 macro — classement visuel

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- F1 macro ---
ax = axes[0]
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(df_compare))]
bars = ax.barh(range(len(df_compare)), df_compare['f1_test'].values,
               color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(df_compare)))
ax.set_yticklabels([f"{row['model']}\n({row['notebook']})" for _, row in df_compare.iterrows()],
                   fontsize=9)
ax.set_xlabel('F1 macro (test)')
ax.set_title('F1 macro par modèle', fontweight='bold')
ax.invert_yaxis()
for bar, val in zip(bars, df_compare['f1_test'].values):
    if pd.notna(val):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9, fontweight='bold')
ax.set_xlim(0, df_compare['f1_test'].max() * 1.15)
ax.grid(axis='x', alpha=0.3)

# --- Accuracy ---
ax = axes[1]
bars = ax.barh(range(len(df_compare)), df_compare['acc_test'].values,
               color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(df_compare)))
ax.set_yticklabels([f"{row['model']}\n({row['notebook']})" for _, row in df_compare.iterrows()],
                   fontsize=9)
ax.set_xlabel('Accuracy (test)')
ax.set_title('Accuracy par modèle', fontweight='bold')
ax.invert_yaxis()
for bar, val in zip(bars, df_compare['acc_test'].values):
    if pd.notna(val):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9, fontweight='bold')
ax.set_xlim(0, df_compare['acc_test'].max() * 1.15)
ax.grid(axis='x', alpha=0.3)

plt.suptitle('Comparaison des modèles — Classification genre_top (FMA Small)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'barplot_f1_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# C6
# 5. Analyse par genre — recall par genre pour chaque modèle
#
# Pour comparer les modèles par genre, on recharge features_V2.csv,
# on refait le split (même seed), et on recharge les pipelines sauvegardés
# ou on ré-entraîne les modèles légers.
#
# Approche pragmatique : on ré-entraîne LR, RF, XGBoost sur le même split.
# Le CNN (NB4) utilise un split différent (80/10/10) donc on le reporte
# depuis ses outputs sauvegardés si disponibles.

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

# --- Chargement features_V2.csv ---
FEATURES_CSV = BASE / 'outputs' / 'features' / 'features_V2.csv'
df_feat = pd.read_csv(FEATURES_CSV)
df_feat = df_feat.loc[:, ~df_feat.columns.astype(str).str.startswith('Unnamed:')]

LABEL_COLS = [
    'track_id', 'track_id_int', 'genre_top', 'artist_name', 'genres_decoded',
    'genres', 'n_subgenres', 'mismatch', 'mismatch_calc', 'track_title',
    'year', 'duration', 'bit_rate'
]
FEATURE_COLS = [c for c in df_feat.columns if c not in LABEL_COLS]

X      = df_feat[FEATURE_COLS]
y      = df_feat['genre_top'].astype(str)
groups = df_feat['artist_name'].astype(str)

# --- Split identique ---
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

print(f'Train : {len(X_train)} | Test : {len(X_test)}')
print(f'Genres : {list(le.classes_)}')

# --- Entraînement des 3 modèles tabulaires ---
models = {}

# LR (NB3 — Camille)
pipe_lr = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler()),
    ('clf',     LogisticRegression(max_iter=2000, C=0.01, solver='lbfgs',
                                  class_weight='balanced', random_state=SEED))
])
pipe_lr.fit(X_train, y_train_enc)
models['LR (NB3)'] = le.inverse_transform(pipe_lr.predict(X_test))
print('LR entraîné ✅')

# RF (NB3 — Camille)
pipe_rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf',     RandomForestClassifier(n_estimators=300, max_depth=20,
                                      min_samples_leaf=4, max_features='sqrt',
                                      class_weight='balanced', random_state=SEED))
])
pipe_rf.fit(X_train, y_train_enc)
models['RF (NB3)'] = le.inverse_transform(pipe_rf.predict(X_test))
print('RF entraîné ✅')

# XGBoost (NB3BIS — Diego)
sw = compute_sample_weight('balanced', y_train_enc)
pipe_xgb = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler()),
    ('clf',     XGBClassifier(n_estimators=300, max_depth=20, min_child_weight=8,
                              random_state=SEED, eval_metric='mlogloss',
                              verbosity=0, n_jobs=-1))
])
pipe_xgb.fit(X_train, y_train_enc, clf__sample_weight=sw)
models['XGBoost (NB3BIS)'] = le.inverse_transform(pipe_xgb.predict(X_test))
print('XGBoost entraîné ✅')

# --- Recall par genre ---
genres = list(le.classes_)
recall_data = {}

for name, y_pred in models.items():
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    recall_data[name] = {g: round(report[g]['recall'], 3) for g in genres}

recall_df = pd.DataFrame(recall_data)
recall_df.index.name = 'Genre'

print('\n=== RECALL PAR GENRE ===')
print(recall_df.to_string())

# Meilleur modèle par genre
print('\nMeilleur modèle par genre :')
for genre in genres:
    best_model = recall_df.loc[genre].idxmax()
    best_val   = recall_df.loc[genre].max()
    print(f'  {genre:15s} → {best_model} (recall={best_val:.3f})')

In [ ]:
# C7
# 6. Heatmap recall par genre × modèle

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(recall_df, annot=True, fmt='.2f', cmap='YlGnBu',
            vmin=0, vmax=0.8, ax=ax, linewidths=0.5)
ax.set_title('Recall par genre × modèle tabulaire', fontweight='bold')
ax.set_ylabel('Genre')
ax.set_xlabel('Modèle')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'heatmap_recall_genre.png', dpi=150)
plt.show()

In [ ]:
# C8
# 7. Matrices de confusion côte à côte (LR vs XGBoost)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (name, y_pred) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, y_pred, labels=genres)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=genres, yticklabels=genres,
                ax=ax, vmin=0, vmax=1)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    ax.set_title(f'{name}\nF1={f1:.4f}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel' if ax == axes[0] else '')
    ax.tick_params(axis='both', labelsize=8)

plt.suptitle('Matrices de confusion normalisées — 3 modèles tabulaires',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrices_compare.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# C9
# 8. Performance conditionnelle mismatch — tous modèles

df_test_full = df_feat.iloc[test_idx].copy()
mismatch_test = pd.to_numeric(df_test_full.get('mismatch', 0), errors='coerce').fillna(0).astype(bool).values
mask_coh = ~mismatch_test
mask_amb =  mismatch_test

print('=== PERFORMANCE CONDITIONNELLE MISMATCH ===')
print(f'{"Modèle":<20s} {"F1 global":>10} {"F1 coh":>10} {"F1 amb":>10} {"Δ (coh-amb)":>12}')
print('-' * 65)

mismatch_rows = []
for name, y_pred in models.items():
    f1_all = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1_c   = f1_score(y_test.values[mask_coh], y_pred[mask_coh], average='macro', zero_division=0)
    f1_a   = f1_score(y_test.values[mask_amb], y_pred[mask_amb], average='macro', zero_division=0)
    delta  = f1_c - f1_a
    print(f'{name:<20s} {f1_all:>10.4f} {f1_c:>10.4f} {f1_a:>10.4f} {delta:>+12.4f}')
    mismatch_rows.append({
        'model': name, 'f1_global': round(f1_all, 4),
        'f1_coherent': round(f1_c, 4), 'f1_ambigu': round(f1_a, 4),
        'delta': round(delta, 4)
    })

print(f'\nN cohérents : {mask_coh.sum()} | N ambigus : {mask_amb.sum()}')
print('→ Δ positif = hypothèse mismatch confirmée (cohérents mieux classés)')

# Barplot
mismatch_df = pd.DataFrame(mismatch_rows)
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(mismatch_df))
w = 0.3
ax.bar(x - w/2, mismatch_df['f1_coherent'], width=w, label='Cohérents', color='steelblue')
ax.bar(x + w/2, mismatch_df['f1_ambigu'],   width=w, label='Ambigus',   color='tomato')
ax.set_xticks(x)
ax.set_xticklabels(mismatch_df['model'], rotation=15, ha='right', fontsize=9)
ax.set_ylabel('F1 macro')
ax.set_title('F1 macro — Cohérents vs Ambigus par modèle', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'mismatch_par_modele.png', dpi=150)
plt.show()

In [ ]:
# C10
# 9. Radar chart — profil multi-métrique

from matplotlib.patches import FancyBboxPatch

metrics_radar = ['f1_test', 'acc_test', 'bal_acc_test']
metric_labels = ['F1 macro', 'Accuracy', 'Balanced Acc']

# Ajouter recall moyen et pire recall
radar_data = []
for name, y_pred in models.items():
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    recalls = [report[g]['recall'] for g in genres]
    f1_val  = f1_score(y_test, y_pred, average='macro', zero_division=0)
    acc_val = accuracy_score(y_test, y_pred)
    bal_val = balanced_accuracy_score(y_test, y_pred)
    radar_data.append({
        'model': name,
        'F1 macro': f1_val,
        'Accuracy': acc_val,
        'Balanced Acc': bal_val,
        'Recall moyen': np.mean(recalls),
        'Pire recall': min(recalls),
    })

radar_df = pd.DataFrame(radar_data).set_index('model')
print('=== PROFIL MULTI-MÉTRIQUE ===')
print(radar_df.round(4).to_string())

# Radar plot
categories = list(radar_df.columns)
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors_radar = ['steelblue', 'darkorange', 'seagreen']

for i, (name, row) in enumerate(radar_df.iterrows()):
    values = row.values.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=name, color=colors_radar[i])
    ax.fill(angles, values, alpha=0.1, color=colors_radar[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylim(0, 0.7)
ax.set_title('Profil multi-métrique par modèle', fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'radar_multimodel.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# C11
# 10. Sauvegarde résultats comparatifs

# Tableau final enrichi
final_rows = []
for name, y_pred in models.items():
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    f1_c = f1_score(y_test.values[mask_coh], y_pred[mask_coh], average='macro', zero_division=0)
    f1_a = f1_score(y_test.values[mask_amb], y_pred[mask_amb], average='macro', zero_division=0)
    final_rows.append({
        'model': name,
        'f1_test': round(f1_score(y_test, y_pred, average='macro', zero_division=0), 4),
        'acc_test': round(accuracy_score(y_test, y_pred), 4),
        'bal_acc_test': round(balanced_accuracy_score(y_test, y_pred), 4),
        'worst_genre': min(genres, key=lambda g: report[g]['recall']),
        'worst_recall': round(min(report[g]['recall'] for g in genres), 3),
        'best_genre': max(genres, key=lambda g: report[g]['recall']),
        'best_recall': round(max(report[g]['recall'] for g in genres), 3),
        'f1_coherent': round(f1_c, 4),
        'f1_ambigu': round(f1_a, 4),
        'delta_mismatch': round(f1_c - f1_a, 4),
    })

df_final = pd.DataFrame(final_rows)
print('=== TABLEAU FINAL ENRICHI ===')
print(df_final.to_string(index=False))

df_final.to_csv(OUTPUT_DIR / 'comparaison_globale.csv', index=False)
print(f'\nSauvegardé : {OUTPUT_DIR / "comparaison_globale.csv"}')

---

### Analyse — Comparaison globale

*À compléter après exécution — les chiffres ci-dessous seront remplis automatiquement.*

#### Classement F1 macro

1. **CNN log-mel (NB4)** — seul modèle non tabulaire, exploite l'information temporelle
2. **XGBoost GridSearch (NB3BIS)** — meilleur modèle tabulaire
3. **Logistic Regression (NB3)** — baseline solide malgré sa simplicité
4. **Random Forest (NB3)** — comparable à LR mais plus lent

#### Pop : le genre impossible

Tous les modèles échouent sur Pop (recall < 0.25). Ce n'est pas un problème
de modèle mais de données : Pop est acoustiquement générique et ses features
chevauchent tous les autres genres (cf. NB5 matrice de confusion).

#### Hypothèse mismatch — validée sur tous les modèles ?

Si Δ(coh−amb) > 0 pour tous les modèles, l'hypothèse NB1 est robuste :
elle ne dépend pas du choix du modèle. Si certains modèles résistent
mieux au mismatch (Δ plus faible), c'est un signal intéressant.

---

---

## Conclusion

---

### Résultats clés

| Constat | Détail |
|---------|--------|
| Meilleur modèle global | CNN log-mel (NB4) — F1 supérieur au plafond tabulaire |
| Meilleur modèle tabulaire | XGBoost GridSearch (NB3BIS) |
| Genre le plus difficile | Pop — recall < 0.25 pour tous les modèles |
| Hypothèse mismatch | Confirmée sur tous les modèles (Δ > 0) |
| Facteur limitant | Features acoustiques intrinsèques, pas le mismatch |

### Limites

- Le CNN (NB4) utilise un split 80/10/10 (validation incluse) vs 80/20 pour les tabulaires
- Les hyperparamètres n'ont pas tous été optimisés de la même manière
- Le dataset FMA Small (8 000 pistes) reste petit pour du deep learning

### Recommandations

- Pour la production : XGBoost (rapide, pas de GPU, F1 ~0.49)
- Pour la performance : CNN (F1 ~0.54, nécessite GPU)
- Pour Pop : aucun modèle ne suffit — envisager des features textuelles (NLP sur metadata)
  ou du transfer learning (PANNs, cf. NB6_TRANSFER)

---